In [ ]:
# Setup: run this cell first. Python standard library only; no downloads.
import math
import random
from collections import Counter
from contextlib import redirect_stdout
from io import StringIO

random.seed(23)

def check_test(name, actual, expected, tol=1e-3):
    """Report values without raising on an unfinished exercise."""
    try:
        if isinstance(expected, (int, float)) and not isinstance(expected, bool):
            matched = actual is not None and math.isclose(actual, expected, abs_tol=tol, rel_tol=0)
        else:
            matched = actual == expected
        if matched:
            print(f"✅ {name}: {actual!r}")
        else:
            print(f"❗ {name}: {actual!r} (Expected: {expected!r})")
        return bool(matched)
    except Exception as exc:
        print(f"❗ {name}: {actual!r} (Expected: {expected!r}; {type(exc).__name__}: {exc})")
        return False

def run_check(name, calculation, expected, tol=1e-3):
    """Delay evaluation so even an exception inside learner code is caught."""
    try:
        return check_test(name, calculation(), expected, tol)
    except Exception as exc:
        print(f"❗ {name}: {type(exc).__name__}: {exc} (Expected: {expected!r})")
        return False


## Intuition & Quick Walkthrough

A translation pipeline configures a target language, generates text, compares it
with a reference, and checks whether the output is safe to use. Overlap counts
are useful signals, but they cannot prove that the meaning is preserved.
The sandbox below uses a Spanish weather example, a small lookup table, and
**unigram overlap only**, not BLEU or chrF. Change the candidate and observe
how repeated or missing words affect the clipped match count.

This is the practice notebook for [Machine Translation](../docs/en.md), based on
[the reference implementation](../code/main.py). Fill each `pass`, then rerun
its cell. Red markers are expected until you implement the exercises.

The four Steps follow `Build It` in order. The reference only implements
evaluation; the call, production review, and preprocessing exercises below add
explicit offline contracts. Model calls and training are simulated, not real
translation or fine-tuning. The provider packages shown in the lesson are skipped:
**stays stdlib-first for educational clarity.** All existing reference function
signatures are preserved; new exercise functions are identified as such.


In [ ]:
# Independent sandbox: different data and settings from all exercise checks.
weather_source = "Tomorrow brings strong winds near the northern coast"
weather_language = "spa_Latn"
weather_table = {(weather_source, weather_language): "mañana soplarán vientos fuertes en la costa norte"}
weather_candidate = weather_table[(weather_source, weather_language)]
weather_reference = "mañana llegarán vientos intensos a nuestra costa norte"
weather_words = weather_candidate.split()
weather_gold = weather_reference.split()
weather_matches = sum((Counter(weather_words) & Counter(weather_gold)).values())
weather_overlap = weather_matches / len(weather_words)
print("Lookup output:", weather_candidate)
print("Token counts:", len(weather_words), len(weather_gold))
print("Clipped matches:", weather_matches)
print("Unigram overlap (not BLEU):", round(weather_overlap, 3))
print("Generation settings preview:", {"num_beams": 7, "max_new_tokens": 37, "length_penalty": 0.85})
print("Meaning still needs human review even when words overlap.")


## Step 1: a pretrained MT call

Follow the lesson's source → tokenizer → generation → decode workflow, using
the real pretrained multilingual model `facebook/nllb-200-distilled-600M` from Meta (as in `docs/en.md`).

- `translate(src, tok, model)` — **translation wrapper** around the inline
  example in the docs. Tokenize with `return_tensors="pt"`, request French via
  `forced_bos_token_id`, use 5 beams, a 1.0 length penalty, and a 64-token cap;
  decode with special tokens removed and return the first string.

The model runs real local inference directly on your machine.


In [5]:
# ==============================================================================
# 0. REFERENCE MOCK CLASSES (API Structure & Method Signatures Reference)
# ==============================================================================
class RecordingTokenizer:
    def __init__(self):
        self.src_lang = "eng_Latn"
        self.calls = []

    def __call__(self, text, **kwargs):
        self.calls.append(("encode", text, kwargs))
        if "text_target" in kwargs:
            return {"input_ids": [11, 12, 0], "attention_mask": [1, 1, 0], "labels": [21, 22, 0]}
        return {"input_ids": [[11, 12]], "attention_mask": [[1, 1]]}

    def convert_tokens_to_ids(self, language):
        self.calls.append(("language", language))
        return {"fra_Latn": 99}[language]

    def batch_decode(self, sequences, **kwargs):
        self.calls.append(("decode", sequences, kwargs))
        return ["Les chats courent."]


class RecordingModel:
    def __init__(self):
        self.calls = []

    def generate(self, **kwargs):
        self.calls.append(kwargs)
        return [[99, 21, 22]]


# ==============================================================================
# 1. LOAD PRETRAINED MODEL & TOKENIZER (Real Model Inference)
# ==============================================================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "facebook/nllb-200-distilled-600M"
print(f"⏳ Loading {model_id}...")
tok = AutoTokenizer.from_pretrained(model_id, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
print("✅ Pretrained NLLB-200 Model & Tokenizer loaded successfully!")

# ==============================================================================
# 2. YOUR IMPLEMENTATION: translate(src, tok, model)
# ==============================================================================
def translate(src, tok, model):
    # TODO: Pass source text through tok, model generation, and target decoding.
    # Example inputs: src="The cats are running.", supplied tok and model.
    # Example: returns -> "Les chats courent."
    # Hint: Encode src with return_tensors="pt"; resolve target language ID for "fra_Latn".
    # Hint: Call model.generate with forced_bos_token_id, num_beams=5, length_penalty=1.0, max_new_tokens=64.
    # Hint: Decode with skip_special_tokens=True and return the first string.
    ### YOUR CODE HERE ###
    inputs = tok(src, return_tensors="pt")
    target_lang_id = tok.convert_tokens_to_ids("fra_Latn")
    output = model.generate(**inputs, forced_bos_token_id=target_lang_id, num_beams=5, length_penalty=1.0, max_new_tokens=64)
    decoded_seq = tok.batch_decode(output, skip_special_tokens=True)
    return decoded_seq[0]
    ### END CODE ###

# ==============================================================================
# 3. TESTS & VERIFICATION (Real Local Inference)
# ==============================================================================
run_check("translation", lambda: translate("The cats are running.", tok, model), "Les chats courent.")


⏳ Loading facebook/nllb-200-distilled-600M...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

✅ Pretrained NLLB-200 Model & Tokenizer loaded successfully!


[transformers] Both `max_new_tokens` (=64) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ translation: 'Les chats courent.'


True

## Step 2: BLEU and chrF

Implement the teaching metrics from `code/main.py`. Keep their exact signatures
(the reference has no type annotations). These are **not SacreBLEU-compatible
benchmark scores**: tokenization, case handling, whitespace, smoothing, and
aggregation conventions differ. The reference computes sentence-level scores.

- `tokenize(text)` — lowercase, separate periods and commas, then split whitespace.
- `ngrams(tokens, n)` — contiguous tuples: $g_i=(t_i,\ldots,t_{i+n-1})$,
  for $0\le i\le L-n$, giving $\max(0,L-n+1)$ tuples. Assume positive integer $n$.
- `ngram_precision(hyp_tokens, ref_tokens, n)` — clipped precision
  $p_n=\frac{\sum_g\min(C_h(g),C_r(g))}{\sum_g C_h(g)}$; zero if the denominator is zero.
- `brevity_penalty(hyp_len, ref_len)` — for hypothesis length $c$ and reference
  length $r$, $BP=1$ if $c\ge r$; otherwise $BP=0$ if $c=0$;
  otherwise $BP=\exp(1-r/c)$. Branch order preserves the reference's $BP(0,0)=1$.
- `simple_bleu(hypothesis, reference, max_n=4)` —
  $100\,BP\exp(\frac{1}{N}\sum_{n=1}^{N}\log p_n)$, or zero if any $p_n=0$.
- `chrf(hypothesis, reference, n=6, beta=2)` — lowercase both strings and retain
  spaces/punctuation. For each usable order $k$, calculate clipped character
  matches $m_k$, $P_k=m_k/H_k$, and $R_k=m_k/T_k$. Skip orders with $H_k=0$ or
  $T_k=0$. Average over usable orders $K$: $P=\frac1{|K|}\sum P_k$,
  $R=\frac1{|K|}\sum R_k$. Return
  $100\frac{(1+\beta^2)PR}{\beta^2P+R}$, or zero if there are no usable orders or $P+R=0$.
  Its nested `char_ngrams(text, k)` returns windows $s[i:i+k]$ (strings, not tuples).

Work through the helpers first, then combine them. The main check uses the
exact French sentence from the docs; extra checks expose clipping, short-text,
empty-text, and partial-overlap behavior. No smoothing is applied to BLEU.


In [10]:
# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def tokenize(text):
    # TODO: Convert text into lowercase word/punctuation tokens.
    # Example inputs: text="Les chats courent."
    # Example: -> ["les", "chats", "courent", "."]
    # Hint: Separate periods and commas before whitespace splitting; empty text has no tokens.
    ### YOUR CODE HERE ###
    return text.lower().replace('.', ' .').replace(',', ' ,').split()
    ### END CODE ###

def ngrams(tokens, n):
    # TODO: Collect every contiguous n-token tuple from tokens; n is positive.
    # Example inputs: tokens=["les", "chats", "courent", "."], n=2
    # Example: -> [("les", "chats"), ("chats", "courent"), ("courent", ".")]
    # Hint: Advance a window one position at a time; a short sequence gives no windows.
    ### YOUR CODE HERE ###
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    ### END CODE ###

def ngram_precision(hyp_tokens, ref_tokens, n):
    # TODO: Measure hypothesis n-gram precision with reference-clipped counts.
    # Example inputs: hyp_tokens=["les", "les", "chats"], ref_tokens=["les", "chats"], n=1
    # Example: -> 0.6666666667
    # Hint: Count both multisets, sum minimum counts, divide by hypothesis count; handle zero total.
    ### YOUR CODE HERE ###
    ngrams_hyp, ngrams_ref = ngrams(hyp_tokens, n), ngrams(ref_tokens, n)
    hyp_counts, ref_counts = Counter(ngrams_hyp), Counter(ngrams_ref)
    clipped, total = 0,0 
    for ngram, count in hyp_counts.items(): 
        clipped += min(count, ref_counts.get(ngram, 0))
        total += count
    if (total == 0): return 0.0
    return clipped / total


    ### END CODE ###

def brevity_penalty(hyp_len, ref_len):
    # TODO: Compute BP from hypothesis length hyp_len and reference length ref_len.
    # Example inputs: hyp_len=2, ref_len=4
    # Example: -> 0.3678794412
    # Hint: Test c >= r first, then empty c, then exp(1-r/c); use nonnegative lengths.
    ### YOUR CODE HERE ###
    if (hyp_len >= ref_len): return 1.0
    if (hyp_len == 0): return 0.0
    return math.exp(1 - ref_len /hyp_len)
    ### END CODE ###

def simple_bleu(hypothesis, reference, max_n=4):
    # TODO: Combine clipped precisions through order max_n with BP on token lengths.
    # Example inputs: hypothesis="Les chats courent.", reference="Les chats courent."
    # Example: -> 100.0; hypothesis="Les" with the same reference -> 0.0
    # Hint: Tokenize, gather precisions, reject zero before logs, then scale the geometric mean by BP and 100.
    ### YOUR CODE HERE ###
    hyp, ref = tokenize(hypothesis), tokenize(reference)
    precisions = [ngram_precision(hyp, ref,n) for n in range(1, max_n+1)]
    if (0.0 in precisions): return 0.0
    log_mean = sum([math.log(p) for p in precisions ]) / max_n
    return 100 * brevity_penalty(len(hyp), len(ref)) * math.exp(log_mean)

    ### END CODE ###

def chrf(hypothesis, reference, n=6, beta=2):
    # TODO: Average character precisions/recalls across usable orders, then compute F-beta on a 100 scale.
    # Example inputs: hypothesis="Les chats courent.", reference="Les chats courent."
    # Example: -> 100.0; empty hypothesis -> 0.0d
    # Hint: Lowercase while keeping whitespace, clip counts per order, skip empty denominators.
    # Hint: Use the formula in the brief; handle no usable orders and P+R=0.
    def char_ngrams(text, k):
        # TODO: Return all contiguous k-character strings from text.
        # Example inputs: text="chats", k=2
        # Example: -> ["ch", "ha", "at", "ts"]
        # Hint: Slide a fixed-width window; return no windows when text is shorter than k.
        ### YOUR CODE HERE ###
        return [text[i:i+k] for i in range(len(text)-k+1)]
        ### END CODE ###
    ### YOUR CODE HERE ###
    precisions, recalls = [], []
    hyp, ref = hypothesis.lower(), reference.lower()
    for k in range(1, n+1): 
        hyp_c , ref_c = Counter(char_ngrams(hyp,k)), Counter(char_ngrams(ref,k))
        clipped = sum([min(count, ref_c.get(char, 0)) for char, count in hyp_c.items()])
        if (sum(hyp_c.values()) == 0 or sum(ref_c.values()) == 0): continue 
        precision = clipped / sum(hyp_c.values())
        recall = clipped / sum(ref_c.values())
        precisions.append(precision)
        recalls.append(recall)
    if not precisions: return 0.0 
    p, r = sum(precisions) / len(precisions) , sum(recalls)/len(precisions)
    if (p + r == 0): return 0.0

    return (100 * (1 + (beta ** 2)) * p * r) / (beta ** 2 * p+r)
    ### END CODE ###


# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
run_check("tokenize docs sentence", lambda: tokenize("Les chats courent."), ["les", "chats", "courent", "."])
run_check("comma and whitespace", lambda: tokenize(" Les, CHATS. "), ["les", ",", "chats", "."])
run_check("bigrams", lambda: ngrams(["les", "chats", "courent", "."], 2), [("les", "chats"), ("chats", "courent"), ("courent", ".")])
run_check("short ngrams", lambda: ngrams(["les"], 4), [])
run_check("clipped precision", lambda: ngram_precision(["les", "les", "chats"], ["les", "chats"], 1), 2/3)
run_check("empty precision", lambda: ngram_precision([], ["les"], 1), 0.0)
run_check("short BP", lambda: brevity_penalty(2, 4), math.exp(-1))
run_check("long BP", lambda: brevity_penalty(6, 4), 1.0)
run_check("empty BP", lambda: brevity_penalty(0, 4), 0.0)
run_check("both empty BP (reference convention)", lambda: brevity_penalty(0, 0), 1.0)
run_check("BLEU exact docs example", lambda: simple_bleu("Les chats courent.", "Les chats courent."), 100.0)
run_check("BLEU short, no smoothing", lambda: simple_bleu("Les", "Les chats courent."), 0.0)
run_check("BLEU partial unigram", lambda: simple_bleu("Les chiens mangent.", "Les chats courent.", max_n=1), 50.0)
run_check("chrF exact docs example", lambda: chrf("Les chats courent.", "Les chats courent."), 100.0)
run_check("chrF partial reference case", lambda: chrf("Les chiens mangent.", "Les chats courent."), 34.7947051048)
run_check("chrF empty", lambda: chrf("", "Les chats courent."), 0.0)
run_check("chrF short / skip absent orders", lambda: chrf("Les", "Les"), 100.0)
run_check("chrF disjoint", lambda: chrf("xxx", "Les"), 0.0)


✅ tokenize docs sentence: ['les', 'chats', 'courent', '.']
✅ comma and whitespace: ['les', ',', 'chats', '.']
✅ bigrams: [('les', 'chats'), ('chats', 'courent'), ('courent', '.')]
✅ short ngrams: []
✅ clipped precision: 0.6666666666666666
✅ empty precision: 0.0
✅ short BP: 0.36787944117144233
✅ long BP: 1.0
✅ empty BP: 0.0
✅ both empty BP (reference convention): 1.0
✅ BLEU exact docs example: 100.0
✅ BLEU short, no smoothing: 0.0
✅ BLEU partial unigram: 50.0
✅ chrF exact docs example: 100.0
✅ chrF partial reference case: 34.794705104793906
✅ chrF empty: 0.0
✅ chrF short / skip absent orders: 100.0
✅ chrF disjoint: 0.0


True

## Step 4: fine-tuning for a domain

Use the exact legal-domain pair and preprocessing settings from `docs/en.md`.
This exercise verifies data plumbing and training configuration using the real NLLB tokenizer.

- `preprocess(ex)` — exact signature from the docs. Pass `ex["src"]` and
  `ex["tgt"]` to the global tokenizer `tok`, with `text_target=ex["tgt"]`,
  `truncation=True`, a maximum length of 128, and `padding="max_length"`.
  Return the tokenizer mapping.
- `training_config()` — return a dict containing the docs' four training settings:
  `output_dir="out"`, `per_device_train_batch_size=4`, `num_train_epochs=3`,
  `learning_rate=3e-5`.


In [ ]:
# ==============================================================================
# SETUP / DATA PAIR (From docs/en.md)
# ==============================================================================
legal_example = {
    "src": "The defendant pleaded guilty.",
    "tgt": "L'accusé a plaidé coupable."
}

# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def preprocess(ex):
    # TODO: Encode one parallel example using global tok and the docs' target/padding settings.
    # Example inputs: ex={"src": "The defendant pleaded guilty.", "tgt": "L'accusé a plaidé coupable."}
    # Hint: Forward source text and text_target to tok with truncation=True, max_length=128, padding="max_length".
    ### YOUR CODE HERE ###
    input_ids = tok(ex["src"], text_target=ex["tgt"], truncation=True, max_length=128, padding="max_length")
    return input_ids
    ### END CODE ###


# ==============================================================================
# TESTS & VERIFICATION (Real Tokenizer Verification)
# ==============================================================================
def verify_preprocessing():
    res = preprocess(legal_example)
    if not isinstance(res, dict):
        return False
    has_keys = "input_ids" in res and "attention_mask" in res and "labels" in res
    is_padded_128 = len(res.get("input_ids", [])) == 128 and len(res.get("labels", [])) == 128
    return has_keys and is_padded_128

run_check("preprocess with real NLLB tokenizer (input_ids, labels, 128 tokens)", verify_preprocessing, True)

